# 07. Pairwise Text Similarity, NLI & Duplicate Matching

> **Kategori**: NLP Competition Starter Template
> **Deskripsi**: Pipeline pemodelan pasangan teks (text_a dan text_b) untuk deteksi pertanyaan duplikat, kemiripan semantik, atau entailment (premise-hypothesis) menggunakan Cross-Encoder Transformer dan Cosine Similarity Features.


### 1. Import Libraries & Setup Environment


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import lightgbm as lgb

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

from tabular_pipeline.text_engine import UniversalTextCleaner
print("Framework ready. ROOT_DIR:", ROOT_DIR)


### 2. Load Dataset & Resolve Text Pair Columns


In [ ]:
# Kolom pasangan teks: teks 1 (misal pertanyaan 1) dan teks 2 (pertanyaan 2)
TRAIN_PATH = "train.csv"

if not os.path.exists(TRAIN_PATH):
    df_train = pd.DataFrame({
        "id": range(1, 101),
        "text_a": ["Bagaimana cara mendaftar beasiswa kampus?"] * 50 + ["Berapa biaya kuliah per semester?"] * 50,
        "text_b": ["Cara pengajuan beasiswa kuliah di kampus"] * 50 + ["Syarat pendaftaran lomba gemastik"] * 50,
        "is_duplicate": [1] * 50 + [0] * 50
    })
else:
    df_train = pd.read_csv(TRAIN_PATH)

COL_A = "text_a"
COL_B = "text_b"
TARGET_COL = "is_duplicate"

print(f"Text Pair Columns: '{COL_A}' & '{COL_B}' | Target: '{TARGET_COL}'")
print(df_train[[COL_A, COL_B, TARGET_COL]].head())


### 3. Feature Engineering Pasangan Teks (Overlap & Cosine Similarity)


In [ ]:
cleaner = UniversalTextCleaner()
df_train["clean_a"] = df_train[COL_A].apply(lambda s: cleaner.clean_minimal(str(s)))
df_train["clean_b"] = df_train[COL_B].apply(lambda s: cleaner.clean_minimal(str(s)))

features = pd.DataFrame(index=df_train.index)

# 1. Selisih panjang karakter dan rasio kata
features["len_diff"] = np.abs(df_train["clean_a"].str.len() - df_train["clean_b"].str.len())
features["word_diff"] = np.abs(df_train["clean_a"].apply(lambda s: len(s.split())) - df_train["clean_b"].apply(lambda s: len(s.split())))

# 2. Jaccard word overlap similarity
def jaccard_similarity(s1, s2):
    w1 = set(s1.lower().split())
    w2 = set(s2.lower().split())
    if not w1 or not w2:
        return 0.0
    return len(w1.intersection(w2)) / len(w1.union(w2))

features["jaccard_overlap"] = [jaccard_similarity(a, b) for a, b in zip(df_train["clean_a"], df_train["clean_b"])]

# 3. TF-IDF Cosine Similarity
corpus = pd.concat([df_train["clean_a"], df_train["clean_b"]]).unique()
vec = TfidfVectorizer(max_features=5000)
vec.fit(corpus)

from sklearn.metrics.pairwise import paired_cosine_distances
X_a = vec.transform(df_train["clean_a"])
X_b = vec.transform(df_train["clean_b"])
features["cosine_sim"] = 1.0 - paired_cosine_distances(X_a, X_b)

print("Generated Pairwise Features:")
print(features.head())


### 4. Cross-Validation Model: LightGBM on Similarity Features


In [ ]:
y = df_train[TARGET_COL].values
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)

oof_lgb = np.zeros(len(df_train))
f1_scores = []

for fold, (trn_idx, val_idx) in enumerate(skf.split(features, y), 1):
    X_tr, y_tr = features.iloc[trn_idx], y[trn_idx]
    X_va, y_va = features.iloc[val_idx], y[val_idx]
    
    clf = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=2026, verbose=-1)
    clf.fit(X_tr, y_tr)
    probs = clf.predict_proba(X_va)[:, 1]
    oof_lgb[val_idx] = probs
    
    preds = (probs >= 0.5).astype(int)
    f1_scores.append(f1_score(y_va, preds, average="macro"))

overall_f1 = f1_score(y, (oof_lgb >= 0.5).astype(int), average="macro")
print(f"Pairwise LightGBM Baseline -> Macro-F1: {overall_f1 * 100:.2f}%")


### 5. Transformer Cross-Encoder Fine-Tuning CLI


In [ ]:
# Jalankan Universal Transformer Cross-Encoder langsung dari terminal atau notebook:
# Format input: text_col dan text_col_2
cmd = f"python {os.path.join(ROOT_DIR, 'train_transformer_universal.py')} --case f1_podium --text_col text_a --text_col_2 text_b --epochs 3"
print("Command for Cross-Encoder Training:")
print(cmd)
